In [ ]:
import datetime as _dt
import html as _html
import itertools
import os

import pandas as pd
from IPython.display import HTML, Image, Markdown
from IPython.display import display as ipy_display
from ugbio_ppmseq.ppmSeq_utils import (
    SORTER_STATS_FORMATS,
    TABLE1_PPMSEQ_FORMATS,
    _assert_adapter_version_supported,
    build_headline_table,
)

In [ ]:
# papermill parameters
sample_name = None
adapter_version = None  # accepted but intentionally not displayed in the report
statistics_h5 = None
strand_ratio_category_png = None
strand_ratio_category_concordance_png = None
sr_hist_png = None
sr_by_et_png = None
read_length_png = None
logo_file = None
illustration_file = None
trimmer_failure_codes_csv = None
sorter_stats_csv = None
extra_info = {}

In [ ]:
if statistics_h5 is None or adapter_version is None or strand_ratio_category_png is None:
    raise ValueError("Missing required input files")
_assert_adapter_version_supported(adapter_version)
sample_name = sample_name or "(unnamed sample)"
# extra_info may arrive as a plain dict from the CLI or as a stringified dict from papermill;
# coerce to a dict either way.
if extra_info is None:
    extra_info = {}
if isinstance(extra_info, str):
    import ast

    extra_info = ast.literal_eval(extra_info) if extra_info else {}
# The sr-only sections are skipped entirely when sr was not present on any read.
sr_plots_present = bool(sr_hist_png) and bool(sr_by_et_png)

In [ ]:
IMAGE_WIDTH = 900

# Monotonic counters — Table 1, Table 2, ... Figure 1, Figure 2, ... in reading order.
_table_counter = itertools.count(1)
_figure_counter = itertools.count(1)


def _next_table():
    return f"Table {next(_table_counter)}"


def _next_figure():
    return f"Figure {next(_figure_counter)}"


def _show_logo(path, align="center"):
    if path and os.path.isfile(path):
        with open(path) as f:
            encoded = f.read().strip()
        ipy_display(
            HTML(
                f'<div style="text-align:{align};margin:12px 0;">'
                f'<img src="data:image/png;base64,{encoded}" style="max-width:320px;height:auto;"/>'
                f"</div>"
            )
        )


def _sample_header():
    # Printed above every table so the reader can always tell which sample the numbers
    # in the row below refer to. sample_name is user-supplied so HTML-escape it.
    ipy_display(HTML(f'<p style="font-weight:bold;margin:2px 0;">' f"Sample: {_html.escape(str(sample_name))}</p>"))


def _caption(label, text):
    # Italic caption line under a figure or table, matching the srsnv report layout.
    ipy_display(HTML(f'<p style="font-style:italic;margin:4px 0 16px 0;"><b>{label}.</b> {text}</p>'))


def _show_image(path, caption_label, caption_text, width=IMAGE_WIDTH):
    if path and os.path.isfile(path):
        ipy_display(Image(path, width=width))
        _caption(caption_label, caption_text)

In [ ]:
_show_logo(logo_file, align="left")
today = _dt.date.today().isoformat()
# Every user-controlled string is HTML-escaped before embedding in the rendered header.
# sample_name and --extra-arg values flow through papermill / CLI, so treat them as untrusted.
_safe_sample = _html.escape(str(sample_name))
header_html = (
    f'<div style="text-align:left;">'
    f'<div style="font-size:14px;color:#888;">{today}</div>'
    f'<div style="font-size:36px;font-weight:bold;margin-top:2px;">ppmSeq QC report</div>'
    f'<div style="font-size:20px;color:#555;margin-top:2px;">Sample: {_safe_sample}</div>'
)
if extra_info:
    kv_rows = "".join(
        f'<tr><td style="padding:2px 12px 2px 0;color:#555;">{_html.escape(str(k))}</td>'
        f'<td style="padding:2px 0;"><code>{_html.escape(str(v))}</code></td></tr>'
        for k, v in extra_info.items()
    )
    header_html += (
        f'<table style="margin:10px 0 0 0;border-collapse:collapse;font-size:14px;">' f"{kv_rows}" f"</table>"
    )
header_html += "</div>"
ipy_display(HTML(header_html))

In [ ]:
toc_lines = [
    "## Table of contents",
    "",
    "- [1. Headline metrics](#1.-Headline-metrics)",
    "- [2. Strand-ratio category plot](#2.-Strand-ratio-category-plot)",
    "- [3. Start / end tag concordance](#3.-Start-/-end-tag-concordance)",
]
if sr_plots_present:
    toc_lines += [
        "- [4. Overall strand-ratio distribution](#4.-Overall-strand-ratio-distribution)",
        "- [5. Strand-ratio by end-tag category](#5.-Strand-ratio-by-end-tag-category)",
    ]
toc_lines += [
    "- [6. Read-length distribution](#6.-Read-length-distribution)",
    "- [7. Full mixed-reads stats](#7.-Full-mixed-reads-stats)",
    "- [8. Full trimming failure metrics](#8.-Full-trimming-failure-metrics)",
    "- [9. Technical details](#9.-Technical-details)",
    "  - [9.1 Run inputs](#9.1-Run-inputs)",
    "  - [9.2 All stored statistics tables](#9.2-All-stored-statistics-tables)",
    "  - [9.3 Read structure](#9.3-Read-structure)",
]
ipy_display(Markdown("\n".join(toc_lines)))

# 1. Headline metrics

In [ ]:
_sample_header()
headline = build_headline_table(statistics_h5).to_frame()
# Apply per-metric formatters. Missing rows are silently skipped (e.g. no trimmer
# failure codes supplied or partial sorter CSV).
per_row_formats = {**TABLE1_PPMSEQ_FORMATS, **SORTER_STATS_FORMATS}


# Build a per-row formatter function the Styler can dispatch on.
def _fmt_row(val, metric):
    fmt = per_row_formats.get(metric)
    if fmt is None:
        return f"{val:.2f}" if isinstance(val, float) else str(val)
    return fmt(val)


# Rendering: build the formatted value column as strings and show as a plain DataFrame
# (the Styler's format dict doesn't support row-dependent callables cleanly when the
# index is the metric name).
formatted = pd.DataFrame(
    {"value": [_fmt_row(headline.loc[m, "value"], m) for m in headline.index]},
    index=headline.index,
)
ipy_display(formatted)
_caption(
    _next_table(),
    "Headline metrics. PCT_MIXED_start_tag: fraction of reads with start-loop tag MIXED. "
    "PCT_failed_adapter_dimers: fraction of reads rejected by Trimmer because the insert was "
    "too short (adapter-dimers). The remaining rows come from the Sorter stats CSV: "
    "PF_Barcode_reads is the total read count; PCT_PF_Reads_aligned is the fraction aligned; "
    "PCT_PF_Q30_bases is the base-quality yield; PCT_SOFTCLIPPED_bases is the fraction of "
    "soft-clipped bases; Mean_Read_Length / Mean_cvg summarise read length and coverage; "
    "PCT_Quality_Trimmed_Reads / PCT_Adapter_Reached / PCT_Chimeras track trimming / adapter "
    "/ chimera rates; Indel_Rate is the per-base indel rate.",
)

# 2. Strand-ratio category plot

In [ ]:
_show_image(
    strand_ratio_category_png,
    _next_figure(),
    "Proportion of reads in each strand-ratio category (MIXED / MINUS / PLUS / UNDETERMINED), "
    "plotted separately for the start- and end-loops. The end-loop bars restrict the denominator "
    "to reads that reached the end loop.",
)

# 3. Start / end tag concordance

In [ ]:
_show_image(
    strand_ratio_category_concordance_png,
    _next_figure(),
    "Concordance heatmap: joint distribution of the start- and end-tag categories across all "
    "reads (top panel) and across only reads where the end was reached (bottom panel). ",
)

In [ ]:
# Section 4 (sr histogram) only appears when the sr tag is present on at least one read.
if sr_plots_present:
    ipy_display(Markdown("# 4. Overall strand-ratio distribution"))
    ipy_display(
        Markdown(
            "The **strand ratio** tag (`sr`) is a per-read float approximately in `[0, 1]`."
            "It measures the ratio of the raw signals consistent with the PLUS strand of the "
            "library template to the signals consistent with both PLUS and MINUS strands. "
            "Reads with `sr ≈ 0` are MINUS-strand; `sr ≈ 1` are PLUS-strand; "
            "intermediate values indicate MIXED inserts (hybridized strands). The `st` tag "
            "is assigned by applying the thresholds `sr ≤ 0.2 → MINUS`, "
            "`sr ≥ 0.8 → PLUS`, else `MIXED`. The histogram below shows the raw `sr` "
            "distribution before that discretization."
        )
    )

In [ ]:
if sr_plots_present:
    _show_image(
        sr_hist_png,
        _next_figure(),
        "Start-tag strand-ratio distribution across all reads, as a line plot over 0.01-wide "
        "bin centers. The sr tag is measured from the ppmSeq start-loop signals. Dashed "
        "vertical lines at 0.2 and 0.8 mark the MIXED read thresholds; values outside [0, 1] are "
        "clipped to the edge bins.",
    )

In [ ]:
if sr_plots_present:
    ipy_display(Markdown("# 5. Strand-ratio by end-tag category"))

In [ ]:
if sr_plots_present:
    _show_image(
        sr_by_et_png,
        _next_figure(),
        "Start-tag strand-ratio distribution faceted by the end-tag category "
        "(MIXED / PLUS / MINUS) on reads whose end was reached. The x-axis value is still "
        "the start-tag sr (the end tag is not captured by sr); faceting by et just slices "
        "reads by their base-called end-tag category. Each panel is normalised to 100% "
        "within its category so the shapes are directly comparable.",
    )

# 6. Read-length distribution

In [ ]:
_show_image(
    read_length_png,
    _next_figure(),
    "Read-length distribution, aligned vs. unaligned reads. Each line is normalised to 1 "
    "within its own group; the legend shows the percent of the subsampled population in each "
    "group. The x-axis is clipped at the 99.5th percentile; bins are 1 bp wide and centered "
    "on integer bp values.",
)

# 7. Full mixed-reads stats

In [ ]:
_sample_header()
stats_shortlist = pd.read_hdf(statistics_h5, key="stats_shortlist")
if isinstance(stats_shortlist, pd.Series):
    stats_shortlist = stats_shortlist.to_frame()
ipy_display(stats_shortlist.style.format("{:.2f}"))
_caption(
    _next_table(),
    "ppmSeq mixed-read metrics, computed only on Trimmer-matched reads (reads with "
    "RG='unmatched' are excluded). PCT_MIXED_start_tag: fraction with start-loop tag MIXED. "
    "PCT_MIXED_both_tags: fraction whose start- and end-loop tags are both MIXED. "
    "The PCT_*_both_tags_where_endreached rows restrict the denominator to reads whose end "
    "loop was reached (tm contains 'A'). PCT_read_end_unreached is the fraction of reads "
    "where the adapter was not reached so no end-tag call is available.",
)

# 8. Full trimming failure metrics

In [ ]:
if trimmer_failure_codes_csv is not None:
    with pd.HDFStore(statistics_h5) as s:
        available_keys = set(s.keys())
    if "/failure_codes_metrics" in available_keys:
        _sample_header()
        df_fc = pd.read_hdf(statistics_h5, key="failure_codes_metrics") / 100.0
        ipy_display(df_fc.style.format("{:.4%}"))
        _caption(
            _next_table(),
            "Trimmer failure-rate metrics, computed as failed_read_count / total_read_count "
            "(denominator is all reads, including unmatched). PCT_failed_adapter_dimers: "
            "fraction rejected because the insert was too short (typically adapter-dimers). "
            "PCT_failed_unrecognized_start_stem: fraction where the start stem sequence "
            "could not be matched. PCT_failed_unrecognized_start_loop: fraction where the "
            "start loop sequence was too long to identify. PCT_failed_total is the overall "
            "fraction of reads Trimmer rejected.",
        )
else:
    ipy_display(HTML("<p><i>No trimmer failure codes CSV supplied.</i></p>"))

# 9. Technical details

## 9.1 Run inputs

In [ ]:
_sample_header()
run_info_rows = [
    ("Sample name", sample_name),
    ("Statistics HDF5", statistics_h5),
    ("Trimmer failure codes CSV", trimmer_failure_codes_csv or "(none supplied)"),
    ("Sorter stats CSV", sorter_stats_csv or "(none supplied)"),
]
for k, v in extra_info.items():
    run_info_rows.append((k, v))
run_info_df = pd.DataFrame(run_info_rows, columns=["field", "value"]).set_index("field")
# Long paths (e.g. "Trimmer failure codes CSV" values) otherwise get visually truncated
# with an ellipsis — force word-wrap so the full path is readable.
html = run_info_df.to_html()
ipy_display(
    HTML(
        "<style>table.dataframe td, table.dataframe th { "
        "word-break: break-all; white-space: normal; max-width: 700px; "
        "text-align: left; vertical-align: top; "
        "}</style>" + html
    )
)
_caption(
    _next_table(),
    "Run inputs for this QC report: sample identifier, auxiliary CSV / HDF5 files, and any "
    "free-form --extra-arg KEY=VALUE pairs (e.g. pipeline version).",
)

## 9.2 All stored statistics tables

In [ ]:
with pd.HDFStore(statistics_h5) as s:
    keys = sorted(s.keys())
shown = 0
for key in keys:
    short = key.lstrip("/")
    # Skip what's already shown or internal-only.
    if short in (
        "stats_shortlist",
        "sorter_stats",
        "failure_codes_metrics",
        "trimmer_failure_codes",
        "subsampled_reads",
        "keys_to_convert",
    ):
        continue
    shown += 1
    _sample_header()
    ipy_display(HTML(f'<p style="font-weight:bold;margin:4px 0;">{short}</p>'))
    ipy_display(pd.read_hdf(statistics_h5, key=key))
if shown == 0:
    ipy_display(HTML("<p><i>No additional stored tables.</i></p>"))
_caption(
    _next_table(),
    "All additional statistics tables stored in the HDF5 bundle beyond those shown earlier. "
    "Typical entries: strand_ratio_category_counts (raw read counts per category), "
    "strand_ratio_category_norm (same as fractions), strand_ratio_category_concordance (pairwise "
    "start/end concordance), strand_ratio_category_consensus (per-read consensus category shares).",
)

## 9.3 Read structure

The diagram below shows where each ppmSeq tag is assigned along the read. The `sr` (strand ratio) is computed by the base caller from raw flow-cell signal before base calling, which is why it is available on all reads (including reads for which the Start / End loop adapters could not be matched).

In [ ]:
if illustration_file is not None and os.path.isfile(illustration_file):
    ipy_display(Image(illustration_file, width=IMAGE_WIDTH))
    _caption(_next_figure(), "ppmSeq read structure and tag assignments.")

In [ ]:
_show_logo(logo_file, align="left")